In [1]:
import os
import re
import html
import warnings
from pathlib import Path

import holidays
import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.corpus import wordnet
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from langdetect import detect, DetectorFactory, LangDetectException

DetectorFactory.seed = 42
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# Konfigurasi path NLTK lokal jika ada
os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"
if os.path.exists("nltk_data"):
    nltk.data.path.insert(0, os.path.abspath("nltk_data"))

# Download resource NLTK yang dibutuhkan (butuh koneksi internet sekali saja)
for pkg in ["stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"Gagal mengunduh '{pkg}': {e}")


In [2]:
news_raw = pd.read_csv('data/raw/cnbc_with_published.csv')
bi_raw = pd.read_csv('data/raw/bi-usd-rate.csv')

print("Berita geopolitik :", news_raw.shape)
print("Kurs USD BI       :", bi_raw.shape)
news_raw.head(5)


Berita geopolitik : (16733, 13)
Kurs USD BI       : (1202, 5)


,keyword_matched,title,url,preview_summary,full_text,date_raw,section,published_raw,published_date,published_time,published_timezone,published_iso,scrape_status
0,geopolitical risk,Global shipping authorities warn of maritime trade breakdown amidgeopoliticalturmoil,https://www.cnbc.com/2026/09/08/global-shipping-iran-war-hormuz-rules.html?&qsearchterm=geopolitical risk,Global maritime authorities have warned that the emergence of “parallel systems” threatens to create a two-tier stru...,"Global maritime authorities have warned that the emergence of ""parallel systems"" threatens to create a two-tier stru...",9/8/2026 2:28:41 PM,Markets,2026-09-08T07:28:41+0000,2026-09-08,07:28 AM,0.0,2026-09-08T07:28:41+0000,ok
1,geopolitical risk,Morning Call Sheet: AI rebound meets risinggeopoliticalrisks,https://www.cnbc.com/video/2026/07/31/morning-call-sheet-ai-rebound-meets-rising-geopolitical-risks.html?&qsearchter...,"Peter Tchir, Head of Macro Strategy at Academy Securities, Thomas Martin, Senior Portfolio Manager at GLOBALT Invest...",NaN,7/31/2026 5:58:53 PM,Morning Call,NaN,NaN,NaN,NaN,NaN,not_found
2,geopolitical risk,"Global markets keep shrugging off shocks. Here’s what could break that streak, according to HSBC",https://www.cnbc.com/2026/09/08/global-markets-shrug-off-shocks-hsbc-sees-what-could-break-the-streak.html?&qsearcht...,"Global markets have shrugged off a barrage of shocks in recent years, but HSBC sees several developments that could ...","In this article Global markets have shrugged off a barrage of shocks in recent years, but HSBC sees several developm...",9/8/2026 11:02:43 AM,World Markets,2026-09-08T04:02:43+0000,2026-09-08,04:02 AM,0.0,2026-09-08T04:02:43+0000,ok
3,geopolitical risk,"Yen hovers near seven-month high, dollar steadies",https://www.cnbc.com/2026/09/08/yen-extends-rally-to-new-seven-month-high-dollar-subdued-ahead-of-cpi.html?&qsearcht...,"The Japanese yen hovered near ​a seven-month high on Tuesday, while the dollar was little changed against major peer...","In this article The Japanese yen hovered near ​a seven-month high on Tuesday, while the dollar was little changed ag...",9/8/2026 11:17:07 AM,Currencies,2026-09-08T04:17:07+0000,2026-09-08,04:17 AM,0.0,2026-09-08T04:17:07+0000,ok
4,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",6/17/2026 2:28:36 PM,Gold,2026-06-17T07:28:36+0000,2026-06-17,07:28 AM,0.0,2026-06-17T07:28:36+0000,ok


In [3]:
news_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16733 entries, 0 to 16732
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   keyword_matched     16733 non-null  object 
 1   title               16733 non-null  object 
 2   url                 16733 non-null  object 
 3   preview_summary     16733 non-null  object 
 4   full_text           13431 non-null  object 
 5   date_raw            16733 non-null  object 
 6   section             16649 non-null  object 
 7   published_raw       15239 non-null  object 
 8   published_date      15239 non-null  object 
 9   published_time      15239 non-null  object 
 10  published_timezone  15239 non-null  float64
 11  published_iso       15239 non-null  object 
 12  scrape_status       16733 non-null  object 
dtypes: float64(1), object(12)
memory usage: 1.7+ MB


In [4]:
# Parse timestamp mentah GEO dan pisahkan tanggal & jam
news_raw["date_raw_dt"] = pd.to_datetime(news_raw["date_raw"], errors="coerce")
news_raw["tanggal"] = news_raw["date_raw_dt"].dt.strftime("%m/%d/%Y")
news_raw["jam"] = news_raw["date_raw_dt"].dt.strftime("%I:%M %p")
news_raw = news_raw.drop(columns=["date_raw"], errors="ignore")

print("Kolom Berita Geopolitik:")
news_raw[["title", "tanggal", "jam", "date_raw_dt"]].head(5)


Kolom Berita Geopolitik:


,title,tanggal,jam,date_raw_dt
0,Global shipping authorities warn of maritime trade breakdown amidgeopoliticalturmoil,09/08/2026,02:28 PM,2026-09-08 14:28:41
1,Morning Call Sheet: AI rebound meets risinggeopoliticalrisks,07/31/2026,05:58 PM,2026-07-31 17:58:53
2,"Global markets keep shrugging off shocks. Here’s what could break that streak, according to HSBC",09/08/2026,11:02 AM,2026-09-08 11:02:43
3,"Yen hovers near seven-month high, dollar steadies",09/08/2026,11:17 AM,2026-09-08 11:17:07
4,Central banks are bringing gold reserves home asgeopoliticalrisks rise,06/17/2026,02:28 PM,2026-06-17 14:28:36


In [5]:
bi_raw["Tanggal_dt"] = pd.to_datetime(bi_raw["Tanggal"], errors="coerce")
bi_raw["tanggal"] = bi_raw["Tanggal_dt"].dt.strftime("%m/%d/%Y")
bi_raw["jam"] = bi_raw["Tanggal_dt"].dt.strftime("%I:%M %p")
bi_raw = bi_raw.drop(
    columns=["Tanggal"],
    errors="ignore"
)

print("Kolom Kurs BI:")
bi_raw.head(5)

Kolom Kurs BI:


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM
3,4,1,17805.58,17628.42,2026-08-27,08/27/2026,12:00 AM
4,5,1,17791.51,17614.49,2026-08-26,08/26/2026,12:00 AM


# Data Cleaning
## Filter Dataset

In [6]:
#delete duplicate rows
def word_count(text):
    if not isinstance(text, str):
        return 0
    return len(text.split())

news = news_raw.copy()
# Ukuran data awal
print(f"Baris awal                         : {len(news_raw)}")

# Filter rentang tanggal inklusif: 1 September 2021 - 1 September 2026
start_date = pd.Timestamp("2021-09-01")
end_date = pd.Timestamp("2026-09-01 23:59:59")
news = news[news["date_raw_dt"].between(start_date, end_date, inclusive="both")]
print(f"Setelah filter rentang 1 September 2021 - 1 September 2026 : {len(news)}")

# Setelah di drop URL
news = news.drop_duplicates(subset=["url"], keep="first")
print(f"Setelah dedup url                  : {len(news)}")

# Setelah di drop title + date_raw
news = news.drop_duplicates(subset=["title", "tanggal", "jam"], keep="first")
print(f"Setelah dedup title+date_raw       : {len(news)}")

Baris awal                         : 16733
Setelah filter rentang 1 September 2021 - 1 September 2026 : 8871
Setelah dedup url                  : 8871
Setelah dedup title+date_raw       : 7084


In [7]:
# Filter full_text yang kosong

# Buang baris yang tidak memiliki isi teks pada ketiga sumber konten.
text_columns = ["title", "preview_summary", "full_text"]
text_content = news[text_columns].fillna("").astype(str).agg(" ".join, axis=1).str.strip()
mask_empty = text_content.eq("")
news = news.loc[~mask_empty].copy()
print(f"Setelah drop teks kosong             : {len(news)}")

# Buang item multimedia berdasarkan URL, judul, dan section.
non_article_pattern = r"\b(video|videos|image|images|photo|photos|foto|gambar|audio|podcast|gallery|galeri|live)\b|\.(mp4|mov|avi|mp3|wav|jpg|jpeg|png)(?:$|[?#])"
content_metadata = news[["url", "title", "section"]].fillna("").astype(str).agg(" ".join, axis=1)
mask_non_article = content_metadata.str.contains(non_article_pattern, case=False, regex=True, na=False)
news = news.loc[~mask_non_article].copy()
print(f"Setelah drop video/gambar/audio      : {len(news)}")

news.head(5)

Setelah drop teks kosong             : 7084
Setelah drop video/gambar/audio      : 6214


,keyword_matched,title,url,preview_summary,full_text,section,published_raw,published_date,published_time,published_timezone,published_iso,scrape_status,date_raw_dt,tanggal,jam
4,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",Gold,2026-06-17T07:28:36+0000,2026-06-17,07:28 AM,0.0,2026-06-17T07:28:36+0000,ok,2026-06-17 14:28:36,06/17/2026,02:28 PM
8,geopolitical risk,Oil rises 2% as White House says no US-Iran talks happening,https://www.cnbc.com/2026/08/27/oil-prices-extend-losses-on-expectations-talks-to-ease-middle-east-supply-woes.html?...,"Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomatic efforts by ot...","In this article Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomat...",Oil,2026-08-27T02:49:03+0000,2026-08-27,02:49 AM,0.0,2026-08-27T02:49:03+0000,ok,2026-08-27 09:49:03,08/27/2026,09:49 AM
9,geopolitical risk,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,https://www.cnbc.com/2026/07/21/jpmorgan-chase-ceo-jamie-dimon-market-risk.html?&qsearchterm=geopolitical risk,JPMorgan Chase CEO Jamie Dimon said investors are underestimating the risks facing the global economy and that he wo...,In this article JPMorgan ChaseCEOJamie Dimonsaid investors are underestimating the risks facing the global economy a...,Finance,2026-07-20T23:01:01+0000,2026-07-20,11:01 PM,0.0,2026-07-20T23:01:01+0000,ok,2026-07-21 06:01:01,07/21/2026,06:01 AM
13,geopolitical risk,"Citi Wealth warns markets may be ‘uncomfortably strong’ amid mountinggeopolitical, inflation risks",https://www.cnbc.com/2026/05/18/citi-wealth-warns-markets-may-be-uncomfortably-strong-as-risks-mount.html?&qsearchte...,"Global markets may be due for a period of consolidation after a sharp rally in equities, even as the longer-term out...",NaN,Pro: Analysis,2026-05-18T05:37:18+0000,2026-05-18,05:37 AM,0.0,2026-05-18T05:37:18+0000,ok,2026-05-18 12:37:18,05/18/2026,12:37 PM
15,geopolitical risk,Where fixed income investors are finding yield asgeopoliticalriskrattles markets,https://www.cnbc.com/2026/04/10/where-fixed-income-investors-are-finding-yield-as-geopolitical-risk-rattles-markets....,"As the Iran war shakes up markets, strategists say there are still plenty of sources of relatively safe yield for in...",NaN,Pro: Income Investing,2026-04-10T19:25:17+0000,2026-04-10,07:25 PM,0.0,2026-04-10T19:25:17+0000,ok,2026-04-11 02:25:17,04/11/2026,02:25 AM


In [8]:
# Panjang text minimal 350

effective_text = news["full_text"].fillna(news["preview_summary"])
news = news.loc[effective_text.apply(word_count) >= 350]
print(f"Setelah filter panjang teks min     : {len(news)}")

Setelah filter panjang teks min     : 4349


In [9]:
#Filter keyword yang relevan dengan geopolitik dan ekonomi
RELEVANCE_KEYWORDS = [
    "geopolitics",
    "geopolitical risk",
    "geopolitical tensions",
    "geopolitical fragmentation",
    "armed conflict",
    "global conflict",
    "international conflict",
    "military tensions",
    "national security",
    "sanctions",
    "trade war",
    "tariffs",
    "embargo",
    "export controls",
    "protectionism",
    "supply chain disruption",
    "diplomacy",
    "foreign policy",
    "nato",
    "brics",
    "opec",
    "g7",
]

EXCLUDE_SECTIONS = [
    "sports", "entertainment", "television", "lifestyle", "travel",
    "food retail", "beer, wine & spirits", "college", "modern medicine",
    "life changes", "fashion", "celebrity",
]

def is_relevant(row):
    """Berita dianggap relevan jika keyword_matched terisi ATAU teks memuat
    salah satu kata kunci geopolitik/ekonomi pada RELEVANCE_KEYWORDS."""
    if isinstance(row["keyword_matched"], str) and row["keyword_matched"].strip():
        return True
    haystack = f"{row.get('title', '')} {row.get('preview_summary', '')}".lower()
    return any(kw in haystack for kw in RELEVANCE_KEYWORDS)

mask_relevant = news.apply(is_relevant, axis=1)
news = news.loc[mask_relevant]

print(f"Setelah filter relevansi kata kunci : {len(news)}")

# 4.5 Filter kategori/section yang tidak relevan
section_lower = news["section"].fillna("").str.lower()
mask_section = ~section_lower.isin(EXCLUDE_SECTIONS)
news = news.loc[mask_section].reset_index(drop=True)

print(f"Setelah filter kategori/section     : {len(news)}")

news.head(3)


Setelah filter relevansi kata kunci : 4349
Setelah filter kategori/section     : 4336


,keyword_matched,title,url,preview_summary,full_text,section,published_raw,published_date,published_time,published_timezone,published_iso,scrape_status,date_raw_dt,tanggal,jam
0,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",Gold,2026-06-17T07:28:36+0000,2026-06-17,07:28 AM,0.0,2026-06-17T07:28:36+0000,ok,2026-06-17 14:28:36,06/17/2026,02:28 PM
1,geopolitical risk,Oil rises 2% as White House says no US-Iran talks happening,https://www.cnbc.com/2026/08/27/oil-prices-extend-losses-on-expectations-talks-to-ease-middle-east-supply-woes.html?...,"Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomatic efforts by ot...","In this article Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomat...",Oil,2026-08-27T02:49:03+0000,2026-08-27,02:49 AM,0.0,2026-08-27T02:49:03+0000,ok,2026-08-27 09:49:03,08/27/2026,09:49 AM
2,geopolitical risk,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,https://www.cnbc.com/2026/07/21/jpmorgan-chase-ceo-jamie-dimon-market-risk.html?&qsearchterm=geopolitical risk,JPMorgan Chase CEO Jamie Dimon said investors are underestimating the risks facing the global economy and that he wo...,In this article JPMorgan ChaseCEOJamie Dimonsaid investors are underestimating the risks facing the global economy a...,Finance,2026-07-20T23:01:01+0000,2026-07-20,11:01 PM,0.0,2026-07-20T23:01:01+0000,ok,2026-07-21 06:01:01,07/21/2026,06:01 AM


In [10]:
# delete missing values
jumlah_nan = news["full_text"].isna().sum()

print("Jumlah full_text NaN:", jumlah_nan)

news_nan = news.loc[news["full_text"].isna()]

news_nan[
    ["url", "title", "preview_summary", "full_text"]
].head()

news = news.dropna(subset=["full_text"]).reset_index(drop=True)

print("Jumlah baris setelah drop NaN:", len(news))
print("Sisa NaN pada full_text:", news["full_text"].isna().sum())

Jumlah full_text NaN: 0
Jumlah baris setelah drop NaN: 4336
Sisa NaN pada full_text: 0


## 2. Data News Cleaning

In [11]:
#menggabungkan title, preview, dan full_text

# Tidak ada nilai kosong yang mengganggu penggabungan teks.
news[["title", "preview_summary", "full_text"]] = (
    news[["title", "preview_summary", "full_text"]].fillna("")
)

# Gabungkan title + preview_summary + full_text.
news["combined_raw"] = (
    ((news["title"] + " ") * 2)
    + news["preview_summary"] + " "
    + news["full_text"]
)

print(f"Jumlah baris: {len(news)}")
news[["title", "combined_raw"]].head(3)

Jumlah baris: 4336


,title,combined_raw
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...
1,Oil rises 2% as White House says no US-Iran talks happening,Oil rises 2% as White House says no US-Iran talks happening Oil rises 2% as White House says no US-Iran talks happen...
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ... Jamie Dimon says...


In [12]:
import unicodedata
from collections import Counter

symbol_counter = Counter()
for text in news["combined_raw"]:
    for ch in text:
        if not (ch.isalnum() or ch.isspace()):
            symbol_counter[ch] += 1

symbol_df = pd.DataFrame(
    [
        (ch, unicodedata.name(ch, "UNKNOWN"), count)
        for ch, count in symbol_counter.most_common()
    ],
    columns=["simbol", "nama_unicode", "jumlah"],
)

print(f"Jumlah jenis simbol unik pada combined_raw: {len(symbol_df)}")
symbol_df

Jumlah jenis simbol unik pada combined_raw: 54


,simbol,nama_unicode,jumlah
0,.,FULL STOP,232989
1,",",COMMA,203423
2,"""",QUOTATION MARK,88912
3,',APOSTROPHE,60351
4,-,HYPHEN-MINUS,41818
5,%,PERCENT SIGN,18915
6,—,EM DASH,10379
7,$,DOLLAR SIGN,9806
8,’,RIGHT SINGLE QUOTATION MARK,6782
9,:,COLON,5100


In [13]:
#cleaning dari tanda baca yang tidak berguna

URL_RE = re.compile(r"http\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
HTML_TAG_RE = re.compile(r"<[^>]+>")
EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U00002700-\U000027BF"
    "\U0001F1E6-\U0001F1FF"
    "\U00002600-\U000026FF"
    "]+",
    flags=re.UNICODE,
)

# Hanya simbol mata uang yang diubah jadi kata.
# Simbol lain (&, +, tanda baca umum) dibuang di NON_ALNUM_RE.
CURRENCY_WORDS = {
    "%": "percent",
    "$": "dollar",
    "£": "pound",
    "€": "euro",
    "¥": "yen",
}
CURRENCY_RE = re.compile("|".join(re.escape(sym) for sym in CURRENCY_WORDS))
NON_ALNUM_RE = re.compile(r"[^a-zA-Z0-9\s]")  # simbol/tanda baca sisa dibuang; huruf & angka dipertahankan
MULTI_SPACE_RE = re.compile(r"\s+")


def normalize_text(text: str) -> str:
    """Hapus HTML/URL/email/emoji, ubah simbol mata uang jadi kata, angka dipertahankan."""
    if not isinstance(text, str) or not text.strip():
        return ""
    t = html.unescape(text)
    t = HTML_TAG_RE.sub(" ", t)
    t = URL_RE.sub(" ", t)
    t = EMAIL_RE.sub(" ", t)
    t = EMOJI_RE.sub(" ", t)
    t = CURRENCY_RE.sub(lambda m: f" {CURRENCY_WORDS[m.group(0)]} ", t)
    t = NON_ALNUM_RE.sub(" ", t)
    t = MULTI_SPACE_RE.sub(" ", t).strip()
    return t


news["normalized_text"] = news["combined_raw"].apply(normalize_text)

news[["combined_raw", "normalized_text"]].head(3)

,combined_raw,normalized_text
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...
1,Oil rises 2% as White House says no US-Iran talks happening Oil rises 2% as White House says no US-Iran talks happen...,Oil rises 2 percent as White House says no US Iran talks happening Oil rises 2 percent as White House says no US Ira...
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ... Jamie Dimon says...,Jamie Dimon says markets underestimate risks and he wouldn t buy stocks or Treasurys at current Jamie Dimon says mar...


In [14]:
# lowercasing
news["lower_text"] = news["normalized_text"].str.lower()

news[["normalized_text", "lower_text"]].head(3)

,normalized_text,lower_text
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...,central banks are bringing gold reserves home asgeopoliticalrisks rise central banks are bringing gold reserves home...
1,Oil rises 2 percent as White House says no US Iran talks happening Oil rises 2 percent as White House says no US Ira...,oil rises 2 percent as white house says no us iran talks happening oil rises 2 percent as white house says no us ira...
2,Jamie Dimon says markets underestimate risks and he wouldn t buy stocks or Treasurys at current Jamie Dimon says mar...,jamie dimon says markets underestimate risks and he wouldn t buy stocks or treasurys at current jamie dimon says mar...


In [15]:
#memastikan bahasanya english semua
def detect_language(text: str) -> str:
    if not isinstance(text, str) or len(text.strip()) < 20:
        return "unknown"
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"


news["detected_lang"] = news["lower_text"].apply(detect_language)

print("Distribusi bahasa terdeteksi:")
print(news["detected_lang"].value_counts().head(10))

baris_sebelum = len(news)
news = news.loc[news["detected_lang"] == "en"].reset_index(drop=True)
print(f"\nSetelah filter bahasa Inggris (en) : {len(news)} (dibuang {baris_sebelum - len(news)} baris)")

news[["lower_text", "detected_lang"]].head(3)

Distribusi bahasa terdeteksi:
detected_lang
en    4336
Name: count, dtype: int64

Setelah filter bahasa Inggris (en) : 4336 (dibuang 0 baris)


,lower_text,detected_lang
0,central banks are bringing gold reserves home asgeopoliticalrisks rise central banks are bringing gold reserves home...,en
1,oil rises 2 percent as white house says no us iran talks happening oil rises 2 percent as white house says no us ira...,en
2,jamie dimon says markets underestimate risks and he wouldn t buy stocks or treasurys at current jamie dimon says mar...,en


## 3. Cleaning Data Kurs

In [16]:
# Gabungkan tanggal dan jam untuk satu kolom datetime kanonik.
bi_raw["Tanggal"] = pd.to_datetime(
    bi_raw["tanggal"].astype(str).str.strip() + " " + bi_raw["jam"].astype(str).str.strip(),
    format="%m/%d/%Y %I:%M %p",
    errors="coerce",
)

print(bi_raw.shape)
bi_raw.head(3)

(1202, 8)


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,Tanggal
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM,2026-09-01
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM,2026-08-31
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM,2026-08-28


In [17]:
def clean_numeric(series: pd.Series) -> pd.Series:
    """Hapus pemisah ribuan dan konversi nilai ke float."""
    cleaned = series.astype(str).str.replace(",", "", regex=False).str.strip()
    return pd.to_numeric(cleaned, errors="coerce")



bi_raw["Kurs Jual"] = clean_numeric(bi_raw["Kurs Jual"])
bi_raw["Kurs Beli"] = clean_numeric(bi_raw["Kurs Beli"])
bi_raw["NO"] = pd.to_numeric(bi_raw["NO"], errors="coerce")
bi_raw["Nilai"] = pd.to_numeric(bi_raw["Nilai"], errors="coerce")

print(f"Baris awal BI                 : {len(bi_raw)}")

# Pertahankan hanya rentang 1 September 2021 sampai 1 September 2026.
bi = bi_raw.loc[bi_raw["Tanggal"].between(start_date, end_date, inclusive="both")]
print(f"Setelah filter rentang tanggal: {len(bi)}")

mask_valid_numeric = (
    bi["Kurs Jual"].notna()
    & bi["Kurs Beli"].notna()
    & (bi["Kurs Jual"] > 0)
    & (bi["Kurs Beli"] > 0)
)
bi = bi.loc[mask_valid_numeric]
print(f"Setelah filter numerik valid  : {len(bi)}")

mask_valid_spread = bi["Kurs Jual"] > bi["Kurs Beli"]
bi = bi.loc[mask_valid_spread]
print(f"Setelah filter Jual > Beli    : {len(bi)}")

bi = (
    bi.sort_values("Tanggal")
    .drop_duplicates(subset=["Tanggal"], keep="last")
    .reset_index(drop=True)
)
print(f"Setelah dedup tanggal         : {len(bi)}")

bi.head(3)

Baris awal BI                 : 1202
Setelah filter rentang tanggal: 1202
Setelah filter numerik valid  : 1202
Setelah filter Jual > Beli    : 1202
Setelah dedup tanggal         : 1202


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,Tanggal
0,1202,1,14377.53,14234.47,2021-09-01,09/01/2021,12:00 AM,2021-09-01
1,1201,1,14355.42,14212.58,2021-09-02,09/02/2021,12:00 AM,2021-09-02
2,1200,1,14352.41,14209.60,2021-09-03,09/03/2021,12:00 AM,2021-09-03


# Preprocessing

In [18]:
#stopword removal
CUSTOM_STOPWORDS = {
    "cnbc", "article", "read", "also", "said", "says", "say",
    "would", "could", "also", "one", "two", "get", "got",
}
STOPWORDS_EN = set(stopwords.words("english")) | CUSTOM_STOPWORDS


def remove_stopwords(text: str) -> list:
    # Tokenisasi split-spasi lalu buang stopword Bahasa Inggris & token 1 karakter.
    if not isinstance(text, str) or not text.strip():
        return []
    tokens = text.split()
    return [tok for tok in tokens if tok not in STOPWORDS_EN and len(tok) > 1]


news["tokens_no_stopwords"] = news["lower_text"].apply(remove_stopwords)
news["text_no_stopwords"] = news["tokens_no_stopwords"].apply(" ".join)

avg_before = news["lower_text"].apply(word_count).mean()
avg_after = news["tokens_no_stopwords"].apply(len).mean()
print(f"Rata-rata jumlah kata sebelum stopword removal : {avg_before:.1f}")
print(f"Rata-rata jumlah kata sesudah stopword removal  : {avg_after:.1f}")

news[["lower_text", "text_no_stopwords"]].head(3)

Rata-rata jumlah kata sebelum stopword removal : 862.4
Rata-rata jumlah kata sesudah stopword removal  : 494.5


,lower_text,text_no_stopwords
0,central banks are bringing gold reserves home asgeopoliticalrisks rise central banks are bringing gold reserves home...,central banks bringing gold reserves home asgeopoliticalrisks rise central banks bringing gold reserves home asgeopo...
1,oil rises 2 percent as white house says no us iran talks happening oil rises 2 percent as white house says no us ira...,oil rises percent white house us iran talks happening oil rises percent white house us iran talks happening oil pric...
2,jamie dimon says markets underestimate risks and he wouldn t buy stocks or treasurys at current jamie dimon says mar...,jamie dimon markets underestimate risks buy stocks treasurys current jamie dimon markets underestimate risks buy sto...


In [19]:
#lemmatization (POS aware)
lemmatizer = WordNetLemmatizer()

POS_MAP = {
    "J": wordnet.ADJ,
    "V": wordnet.VERB,
    "N": wordnet.NOUN,
    "R": wordnet.ADV,
}


def get_wordnet_pos(treebank_tag: str) -> str:
    # Tag POS Penn Treebank (huruf pertama) ke format POS WordNet; default kata benda.
    return POS_MAP.get(treebank_tag[0], wordnet.NOUN)


def lemmatize_tokens(tokens: list) -> list:
    # Lemmatisasi tiap token berdasarkan POS tag-nya (POS-aware lemmatization).
    if not tokens:
        return []
    tagged_tokens = pos_tag(tokens)
    return [
        lemmatizer.lemmatize(token, pos=get_wordnet_pos(tag))
        for token, tag in tagged_tokens
    ]


news["tokens_lemmatized"] = news["tokens_no_stopwords"].apply(lemmatize_tokens)
news["text_lemmatized"] = news["tokens_lemmatized"].apply(" ".join)

news[["text_no_stopwords", "text_lemmatized"]].head(3)


,text_no_stopwords,text_lemmatized
0,central banks bringing gold reserves home asgeopoliticalrisks rise central banks bringing gold reserves home asgeopo...,central bank bring gold reserve home asgeopoliticalrisks rise central bank bring gold reserve home asgeopoliticalris...
1,oil rises percent white house us iran talks happening oil rises percent white house us iran talks happening oil pric...,oil rise percent white house u iran talk happen oil rise percent white house u iran talk happen oil price rise thurs...
2,jamie dimon markets underestimate risks buy stocks treasurys current jamie dimon markets underestimate risks buy sto...,jamie dimon market underestimate risk buy stock treasurys current jamie dimon market underestimate risk buy stock tr...


In [20]:
news

,keyword_matched,title,url,preview_summary,full_text,section,published_raw,published_date,published_time,published_timezone,...,tanggal,jam,combined_raw,normalized_text,lower_text,detected_lang,tokens_no_stopwords,text_no_stopwords,tokens_lemmatized,text_lemmatized
0,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",Gold,2026-06-17T07:28:36+0000,2026-06-17,07:28 AM,0.0,...,06/17/2026,02:28 PM,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...,central banks are bringing gold reserves home asgeopoliticalrisks rise central banks are bringing gold reserves home...,en,"[central, banks, bringing, gold, reserves, home, asgeopoliticalrisks, rise, central, banks, bringing, gold, reserves...",central banks bringing gold reserves home asgeopoliticalrisks rise central banks bringing gold reserves home asgeopo...,"[central, bank, bring, gold, reserve, home, asgeopoliticalrisks, rise, central, bank, bring, gold, reserve, home, as...",central bank bring gold reserve home asgeopoliticalrisks rise central bank bring gold reserve home asgeopoliticalris...
1,geopolitical risk,Oil rises 2% as White House says no US-Iran talks happening,https://www.cnbc.com/2026/08/27/oil-prices-extend-losses-on-expectations-talks-to-ease-middle-east-supply-woes.html?...,"Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomatic efforts by ot...","In this article Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomat...",Oil,2026-08-27T02:49:03+0000,2026-08-27,02:49 AM,0.0,...,08/27/2026,09:49 AM,Oil rises 2% as White House says no US-Iran talks happening Oil rises 2% as White House says no US-Iran talks happen...,Oil rises 2 percent as White House says no US Iran talks happening Oil rises 2 percent as White House says no US Ira...,oil rises 2 percent as white house says no us iran talks happening oil rises 2 percent as white house says no us ira...,en,"[oil, rises, percent, white, house, us, iran, talks, happening, oil, rises, percent, white, house, us, iran, talks, ...",oil rises percent white house us iran talks happening oil rises percent white house us iran talks happening oil pric...,"[oil, rise, percent, white, house, u, iran, talk, happen, oil, rise, percent, white, house, u, iran, talk, happen, o...",oil rise percent white house u iran talk happen oil rise percent white house u iran talk happen oil price rise thurs...
2,geopolitical risk,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,https://www.cnbc.com/2026/07/21/jpmorgan-chase-ceo-jamie-dimon-market-risk.html?&qsearchterm=geopolitical risk,JPMorgan Chase CEO Jamie Dimon said investors are underestimating the risks facing the global economy and that he wo...,In this article JPMorgan ChaseCEOJamie Dimonsaid investors are underestimating the risks facing the global economy a...,Finance,2026-07-20T23:01:01+0000,2026-07-20,11:01 PM,0.0,...,07/21/2026,06:01 AM,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ... Jamie Dimon says...,Jamie Dimon says markets underestimate risks and he wouldn t buy stocks or Treasurys at current Jamie Dimon says mar...,jamie dimon says markets underestimate risks and he wouldn t buy stocks or treasurys at current jamie dimon says mar...,en,"[jamie, dimon, markets, underestimate, risks, buy, stocks, treasurys, current

# Penyelerasan Data

In [21]:
#change timezone to WIB
news['published_raw'] = pd.to_datetime(news['published_raw'], errors='coerce', utc=True)
news['published_raw'] = news['published_raw'].dt.tz_convert('Asia/Jakarta')

#pisahin tanggal & waktu
news['date'] = news['published_raw'].dt.date
news['time'] = news['published_raw'].dt.time

In [22]:
def align_to_trading_date(dt):
    if pd.isna(dt):
        return pd.NaT
    # Step 1: kalau lewat cutoff 15:00:00, geser ke hari berikutnya
    cutoff = dt.replace(hour=15, minute=0, second=0, microsecond=0)
    if dt > cutoff:
        dt = dt + pd.Timedelta(days=1)

    # Step 2: kalau hasilnya jatuh di weekend, geser ke Senin
    if dt.weekday() == 5:      # Sabtu
        dt = dt + pd.Timedelta(days=2)
    elif dt.weekday() == 6:    # Minggu
        dt = dt + pd.Timedelta(days=1)

    return dt.normalize()  # buang jam, sisakan tanggal saja

news['aligned_trading_date'] = news['date_raw_dt'].apply(align_to_trading_date)

# =========================================================================
# EXPORT CSV: DATA BERITA SEBELUM LEMMATIZE DKK (TAPI SUDAH ALIGNED)
# =========================================================================
import os
os.makedirs('data/processed', exist_ok=True)

# Kolom teks bersih asli (sebelum stopword removal & lemmatization)
cols_unprocessed = [
    c for c in ['url', 'title', 'preview_summary', 'full_text', 'combined_raw', 
                'normalized_text', 'lower_text', 'date_raw_dt', 'aligned_trading_date']
    if c in news.columns
]
news_aligned_unprocessed = news[cols_unprocessed].copy()

unprocessed_csv_path = 'data/processed/news_aligned_unprocessed.csv'
news_aligned_unprocessed.to_csv(unprocessed_csv_path, index=False)

print("=" * 75)
print("BERHASIL MENYIMPAN CSV (BELUM LEMMATIZE DKK, TAPI SUDAH ALIGNED):")
print(f"Path   : {unprocessed_csv_path}")
print(f"Ukuran : {news_aligned_unprocessed.shape[0]} baris x {news_aligned_unprocessed.shape[1]} kolom")
print(f"Kolom  : {list(news_aligned_unprocessed.columns)}")
print("=" * 75)
news_aligned_unprocessed.head(3)


BERHASIL MENYIMPAN CSV (BELUM LEMMATIZE DKK, TAPI SUDAH ALIGNED):
Path   : data/processed/news_aligned_unprocessed.csv
Ukuran : 4336 baris x 9 kolom
Kolom  : ['url', 'title', 'preview_summary', 'full_text', 'combined_raw', 'normalized_text', 'lower_text', 'date_raw_dt', 'aligned_trading_date']


,url,title,preview_summary,full_text,combined_raw,normalized_text,lower_text,date_raw_dt,aligned_trading_date
0,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...,central banks are bringing gold reserves home asgeopoliticalrisks rise central banks are bringing gold reserves home...,2026-06-17 14:28:36,2026-06-17
1,https://www.cnbc.com/2026/08/27/oil-prices-extend-losses-on-expectations-talks-to-ease-middle-east-supply-woes.html?...,Oil rises 2% as White House says no US-Iran talks happening,"Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomatic efforts by ot...","In this article Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomat...",Oil rises 2% as White House says no US-Iran talks happening Oil rises 2% as White House says no US-Iran talks happen...,Oil rises 2 percent as White House says no US Iran talks happening Oil rises 2 percent as White House says no US Ira...,oil rises 2 percent as white house says no us iran talks happening oil rises 2 percent as white house says no us ira...,2026-08-27 09:49:03,2026-08-27
2,https://www.cnbc.com/2026/07/21/jpmorgan-chase-ceo-jamie-dimon-market-risk.html?&qsearchterm=geopolitical risk,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,JPMorgan Chase CEO Jamie Dimon said investors are underestimating the risks facing the global economy and that he wo...,In this article JPMorgan ChaseCEOJamie Dimonsaid investors are underestimating the risks facing the global economy a...,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ... Jamie Dimon says...,Jamie Dimon says markets underestimate risks and he wouldn t buy stocks or Treasurys at current Jamie Dimon says mar...,jamie dimon says markets underestimate risks and he wouldn t buy stocks or treasurys at current jamie dimon says mar...,2026-07-21 06:01:01,2026-07-21


# VADER

In [23]:
# urutkan dulu supaya urutan artikel konsisten antar kolom
news_sorted = news.sort_values('date_raw_dt')

# semua kolom teks (kecuali kolom tanggal/kunci)
text_cols = [
    c for c in news_sorted.select_dtypes(include=['object', 'string']).columns
    if c not in ['aligned_trading_date']
]

def join_pipe(x):
    return ' | '.join(x.fillna('').astype(str))

agg_spec = {'news_count': ('title', 'size')}   # size = ikut hitung NaN
for c in text_cols:
    agg_spec[f'all_{c}'] = (c, join_pipe)

daily_news = (
    news_sorted.groupby('aligned_trading_date')
               .agg(**agg_spec)
               .reset_index()
)

# rata-ratakan nilai statistik VADER untuk 1 tanggal aligned

daily_news

,aligned_trading_date,news_count,all_keyword_matched,all_title,all_url,all_preview_summary,all_full_text,all_section,all_published_date,all_published_time,...,all_combined_raw,all_normalized_text,all_lower_text,all_detected_lang,all_tokens_no_stopwords,all_text_no_stopwords,all_tokens_lemmatized,all_text_lemmatized,all_date,all_time
0,2021-09-06,1,global conflict,"Inflation could repeat the 1960s, when the Fed lost control, Niall Ferguson says",https://www.cnbc.com/2021/09/03/niall-ferguson-inflation-could-repeat-the-1960s-when-the-fed-lost-control.html?&qsea...,"Inflation could be repeating the trajectory of the late 1960s, which laid the foundation for sustained high prices t...","Inflation could be repeating the trajectory of the late 1960s, which laid the foundation for sustained high prices t...",Economy,2021-09-03,10:07 AM,...,"Inflation could repeat the 1960s, when the Fed lost control, Niall Ferguson says Inflation could repeat the 1960s, w...",Inflation could repeat the 1960s when the Fed lost control Niall Ferguson says Inflation could repeat the 1960s when...,inflation could repeat the 1960s when the fed lost control niall ferguson says inflation could repeat the 1960s when...,en,"['inflation', 'repeat', '1960s', 'fed', 'lost', 'control', 'niall', 'ferguson', 'inflation', 'repeat', '1960s', 'fed...",inflation repeat 1960s fed lost control niall ferguson inflation repeat 1960s fed lost control niall ferguson inflat...,"['inflation', 'repeat', '1960s', 'fed', 'lose', 'control', 'niall', 'ferguson', 'inflation', 'repeat', '1960s', 'fed...",inflation repeat 1960s fed lose control niall ferguson inflation repeat 1960s fed lose control niall ferguson inflat...,2021-09-03,17:07:08
1,2021-09-08,1,geopolitical risk,"Guinea coup rattles iron ore and bauxite markets, stokes economic uncertainty",https://www.cnbc.com/2021/09/07/guinea-coup-rattles-iron-ore-and-bauxite-markets-stokes-economic-uncertainty.html?&q...,A military junta claimed to have seized control in the West African country of Guinea and detained President Alpha C...,A military juntaclaimed to have seized controlin the West African country of Guinea and detained President Alpha Con...,World Politics,2021-09-07,10:37 AM,...,"Guinea coup rattles iron ore and bauxite markets, stokes economic uncertainty Guinea coup rattles iron ore and bauxi...",Guinea coup rattles iron ore and bauxite markets stokes economic uncertainty Guinea coup rattles iron ore and bauxit...,guinea coup rattles iron ore and bauxite markets stokes economic uncertainty guinea coup rattles iron ore and bauxit...,en,"['guinea', 'coup', 'rattles', 'iron', 'ore', 'bauxite', 'markets', 'stokes', 'economic', 'uncertainty', 'guinea', 'c...",guinea coup rattles iron ore bauxite markets stokes economic uncertainty guinea coup rattles iron ore bauxite market...,"['guinea', 'coup', 'rattle', 'iron', 'ore', 'bauxite', 'market', 'stoke', 'economic', 'uncertainty', 'guinea', 'coup...",guinea coup rattle iron ore bauxite market stoke economic uncertainty guinea coup rattle iron ore bauxite market sto...,2021-09-07,17:37:21
2,2021-09-09,1,diplomacy,Op-ed: A key choice awaits China — money or principles,https://www.cnbc.com/2021/09/08/op-ed-a-key-choice-awaits-china-money-or-principles.html?&qsearchterm=diplomacy,China is at a crucial moment in its political and economic evolution. The political growl that is trying to become a...,China is at a crucial moment in its political and economic evolution. The political growl that is trying to become a...,Markets,2021-09-08,06:11 PM,...,Op-ed: A key choice awaits China — money or principles Op-ed: A key choice awaits China — money or principles China ...,Op ed A key choice awaits China money or principles Op ed A key choice awaits China money or principles China is at ...,op ed a key choice awaits china money or principles op ed a key choice awaits china money or principles china is at ...,en,"['op', 'ed', 'key', 'choice', 'awaits', 'china', 'money', 'principles', 'op',

In [24]:
# 1. Samakan tipe & buang jam di kedua kunci
bi_raw['Tanggal_dt'] = pd.to_datetime(bi_raw['Tanggal_dt']).dt.normalize()
daily_news['aligned_trading_date'] = pd.to_datetime(daily_news['aligned_trading_date']).dt.normalize()

# 2. Cek duplikat kunci (kalau ada, baris akan membengkak setelah merge)
print(bi_raw['Tanggal_dt'].duplicated().sum())
print(daily_news['aligned_trading_date'].duplicated().sum())

# 3. Merge
merged_data = pd.merge(
    bi_raw,
    daily_news,
    left_on='Tanggal_dt',
    right_on='aligned_trading_date',
    how='left',
    validate='one_to_one'   # error kalau ada duplikat, jadi ketahuan
).drop(columns='aligned_trading_date')  # kolom kunci kembar, tidak perlu

# 4. Hari tanpa berita: isi supaya tidak NaN
merged_data['news_count'] = merged_data['news_count'].fillna(0).astype(int)
text_cols = [c for c in merged_data.columns if c.startswith('all_')]
merged_data[text_cols] = merged_data[text_cols].fillna('')

merged_data

0
0


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,Tanggal,news_count,all_keyword_matched,...,all_combined_raw,all_normalized_text,all_lower_text,all_detected_lang,all_tokens_no_stopwords,all_text_no_stopwords,all_tokens_lemmatized,all_text_lemmatized,all_date,all_time
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM,2026-09-01,1,global conflict,...,Oil prices rise as latest fighting resurrects Middle East supply disruption risks Oil prices rise as latest fighting...,Oil prices rise as latest fighting resurrects Middle East supply disruption risks Oil prices rise as latest fighting...,oil prices rise as latest fighting resurrects middle east supply disruption risks oil prices rise as latest fighting...,en,"['oil', 'prices', 'rise', 'latest', 'fighting', 'resurrects', 'middle', 'east', 'supply', 'disruption', 'risks', 'oi...",oil prices rise latest fighting resurrects middle east supply disruption risks oil prices rise latest fighting resur...,"['oil', 'price', 'rise', 'late', 'fight', 'resurrects', 'middle', 'east', 'supply', 'disruption', 'risk', 'oil', 'pr...",oil price rise late fight resurrects middle east supply disruption risk oil price rise late fight resurrects middle ...,2026-09-01,10:13:07
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM,2026-08-31,7,armed conflict | global conflict | geopolitical tensions | diplomacy | global conflict | global conflict | armed con...,...,Russian forces intensify attacks in Donetsk as Ukraine lauds fresh EU push to unlock frozen ... Russian forces inten...,Russian forces intensify attacks in Donetsk as Ukraine lauds fresh EU push to unlock frozen Russian forces intensify...,russian forces intensify attacks in donetsk as ukraine lauds fresh eu push to unlock frozen russian forces intensify...,en | en | en | en | en | en | en,"['russian', 'forces', 'intensify', 'attacks', 'donetsk', 'ukraine', 'lauds', 'fresh', 'eu', 'push', 'unlock', 'froze...",russian forces intensify attacks donetsk ukraine lauds fresh eu push unlock frozen russian forces intensify attacks ...,"['russian', 'force', 'intensify', 'attack', 'donetsk', 'ukraine', 'lauds', 'fresh', 'eu', 'push', 'unlock', 'frozen'...",russian force intensify attack donetsk ukraine lauds fresh eu push unlock frozen russian force intensify attack done...,2026-08-28 | 2026-08-29 | 2026-08-30 | 2026-08-30 | 2026-08-31 | 2026-08-31 | 2026-08-31,16:33:45 | 19:00:01 | 20:10:03 | 20:10:03 | 04:32:59 | 13:59:11 | 14:29:27
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM,2026-08-28,6,geopolitical risk | geopolitical risk | global conflict | diplomacy | geopolitical risk | geopolitical risk,...,China’s super-rich fled Singapore. Now they want to come back China’s super-rich fled Singapore. Now they want to co...,China s super rich fled Singapore Now they want to come back China s super rich fled Singapore Now they want to come...,china s super rich fled singapore now they want to come back china s super rich fled singapore now they want to come...,en | en | en | en | en | en,"['china', 'super', 'rich', 'fled', 'singapore', 'want', 'come', 'back', 'china', 'super', 'rich', 'fled', 'singapore...",china super rich fled singapore want come back china super rich fled singapore want come back year ago wealthy chine...,"['china', 'super', 'rich', 'flee', 'singapore', 'want', 'come', 'back', 'china', 'super', 'rich', 'flee', 'singapore...",china super rich flee singapore want come back china super rich flee singapore want come back year ago wealthy chine...,2026-08-28 | 2026-08-28 | 2026-08-28 | 2026-08-28 | 2026-08-28 | 2026-08-28,06:29:56 | 09:05:10 | 09:29:22 | 09:29:22 | 12:00:01 | 12:02:16
3,4,1,17805.58,17628.42,2026-08-27,08/27/2026,12:00 AM,2026-08-27,1,geopolitical risk,...,Oil rises 2% as White House says no US-Iran talks happening Oil rises 2% as White House says no US-Iran talks happen...,Oil rises 2 percent as White House says no US Iran talks happening Oil rises 2 percent as White House says no US Ira...,oil rises 2 percen

In [25]:
# drop column
COL = ["Nilai", "Tanggal","all_url","all_section","all_published_date","all_published_time","all_published_iso","all_scrape_status","all_tanggal","all_jam","all_detected_lang","all_date","all_time"]

merged_data_clean = merged_data.drop(columns=COL, errors="ignore")
merged_data_clean

,NO,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,news_count,all_keyword_matched,all_title,all_preview_summary,all_full_text,all_combined_raw,all_normalized_text,all_lower_text,all_tokens_no_stopwords,all_text_no_stopwords,all_tokens_lemmatized,all_text_lemmatized
0,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM,1,global conflict,Oil prices rise as latest fighting resurrects Middle East supply disruption risks,Oil prices gained on Tuesday as the ​resumption of fighting between the U.S. and Iran in the Middle East renewed fea...,In this article Oil prices gained on Tuesday as the ​resumption of fighting between the U.S. and Iran in the Middle ...,Oil prices rise as latest fighting resurrects Middle East supply disruption risks Oil prices rise as latest fighting...,Oil prices rise as latest fighting resurrects Middle East supply disruption risks Oil prices rise as latest fighting...,oil prices rise as latest fighting resurrects middle east supply disruption risks oil prices rise as latest fighting...,"['oil', 'prices', 'rise', 'latest', 'fighting', 'resurrects', 'middle', 'east', 'supply', 'disruption', 'risks', 'oi...",oil prices rise latest fighting resurrects middle east supply disruption risks oil prices rise latest fighting resur...,"['oil', 'price', 'rise', 'late', 'fight', 'resurrects', 'middle', 'east', 'supply', 'disruption', 'risk', 'oil', 'pr...",oil price rise late fight resurrects middle east supply disruption risk oil price rise late fight resurrects middle ...
1,2,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM,7,armed conflict | global conflict | geopolitical tensions | diplomacy | global conflict | global conflict | armed con...,Russian forces intensify attacks in Donetsk as Ukraine lauds fresh EU push to unlock frozen ... | Goodyear burning r...,Russian forces have intensified attacks within Ukraine’s so-called “fortress belt” of industrial cities in the easte...,"Russian forces have intensified attacks within Ukraine's so-called ""fortress belt"" of industrial cities in the easte...",Russian forces intensify attacks in Donetsk as Ukraine lauds fresh EU push to unlock frozen ... Russian forces inten...,Russian forces intensify attacks in Donetsk as Ukraine lauds fresh EU push to unlock frozen Russian forces intensify...,russian forces intensify attacks in donetsk as ukraine lauds fresh eu push to unlock frozen russian forces intensify...,"['russian', 'forces', 'intensify', 'attacks', 'donetsk', 'ukraine', 'lauds', 'fresh', 'eu', 'push', 'unlock', 'froze...",russian forces intensify attacks donetsk ukraine lauds fresh eu push unlock frozen russian forces intensify attacks ...,"['russian', 'force', 'intensify', 'attack', 'donetsk', 'ukraine', 'lauds', 'fresh', 'eu', 'push', 'unlock', 'frozen'...",russian force intensify attack donetsk ukraine lauds fresh eu push unlock frozen russian force intensify attack done...
2,3,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM,6,geopolitical risk | geopolitical risk | global conflict | diplomacy | geopolitical risk | geopolitical risk,China’s super-rich fled Singapore. Now they want to come back | Oil prices fall for the week on U.S.-Iran talks stal...,"A year ago, wealthy Chinese families were souring on Singapore. Its rules felt onerous, its nightlife subdued. Other...","A year ago, wealthy Chinese familieswere souring on Singapore. Its rules felt onerous, its nightlife subdued. Other ...",China’s super-rich fled Singapore. Now they want to come back China’s super-rich fled Singapore. Now they want to co...,China s super rich fled Singapore Now they want to come back China s super rich fled Singapore Now they want to come...,china s super rich fled singapore now they want to come back china s super rich fled singapore now they want to come...,"['china', 'super', 'rich', 'fled', 'singapore', 'want', 'come', 'back', 'china', 'super', 'rich', 'fled', 'singapore...",china super rich fled singapore want come back china super rich fled singapore want come back year ago wealth

In [26]:
# =========================================================================
# EXPORT CSV: DATA HASIL PREPROCESSING LENGKAP
# =========================================================================
import os
os.makedirs('data/processed', exist_ok=True)

# 1. Simpan CSV Berita Artikel yang SUDAH DIPROSES tapi TIDAK DIMERGE (Level Artikel)
cols_unmerged = [
    c for c in ['url', 'title', 'preview_summary', 'full_text', 'combined_raw',
                'normalized_text', 'lower_text', 'text_no_stopwords', 'text_lemmatized',
                'date_raw_dt', 'tanggal', 'jam', 'aligned_trading_date']
    if c in news.columns
]
news_processed_unmerged = news[cols_unmerged].copy()
unmerged_csv_path = 'data/processed/news_processed_unmerged.csv'
news_processed_unmerged.to_csv(unmerged_csv_path, index=False)

# 2. Simpan CSV Final Clean (Sudah Diproses + Sudah Dimerge Kurs BI harian)
clean_csv_path = 'data/processed/merged_data_clean.csv'
merged_data_clean.to_csv(clean_csv_path, index=False)

# 3. Simpan juga CSV Daily Merged yang khusus teks sebelum lemmatize
cols_unprocessed_daily = [
    c for c in merged_data_clean.columns 
    if c not in ['all_tokens_no_stopwords', 'all_text_no_stopwords', 'all_tokens_lemmatized', 'all_text_lemmatized']
]
daily_unprocessed_path = 'data/processed/merged_data_unprocessed.csv'
merged_data_clean[cols_unprocessed_daily].to_csv(daily_unprocessed_path, index=False)

print("=" * 75)
print("BERHASIL MENYIMPAN OUTPUT CSV:")
print(f"1. Artikel Diproses (TIDAK DIMERGE)       : {unmerged_csv_path}")
print(f"   Ukuran : {news_processed_unmerged.shape[0]} baris x {news_processed_unmerged.shape[1]} kolom")
print(f"2. Final Clean (Sudah Lemmatize + Dimerge): {clean_csv_path}")
print(f"   Ukuran : {merged_data_clean.shape[0]} baris x {merged_data_clean.shape[1]} kolom")
print(f"3. Daily Merged (Belum Lemmatize + Dimerge): {daily_unprocessed_path}")
print(f"   Ukuran : {merged_data_clean[cols_unprocessed_daily].shape[0]} baris x {len(cols_unprocessed_daily)} kolom")
print("=" * 75)
print("\nPreview 3 baris teratas news_processed_unmerged:")
display(news_processed_unmerged.head(3))


BERHASIL MENYIMPAN OUTPUT CSV:
1. Artikel Diproses (TIDAK DIMERGE)       : data/processed/news_processed_unmerged.csv
   Ukuran : 4336 baris x 13 kolom
2. Final Clean (Sudah Lemmatize + Dimerge): data/processed/merged_data_clean.csv
   Ukuran : 1202 baris x 18 kolom
3. Daily Merged (Belum Lemmatize + Dimerge): data/processed/merged_data_unprocessed.csv
   Ukuran : 1202 baris x 14 kolom

Preview 3 baris teratas news_processed_unmerged:


,url,title,preview_summary,full_text,combined_raw,normalized_text,lower_text,text_no_stopwords,text_lemmatized,date_raw_dt,tanggal,jam,aligned_trading_date
0,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...,central banks are bringing gold reserves home asgeopoliticalrisks rise central banks are bringing gold reserves home...,central banks bringing gold reserves home asgeopoliticalrisks rise central banks bringing gold reserves home asgeopo...,central bank bring gold reserve home asgeopoliticalrisks rise central bank bring gold reserve home asgeopoliticalris...,2026-06-17 14:28:36,06/17/2026,02:28 PM,2026-06-17
1,https://www.cnbc.com/2026/08/27/oil-prices-extend-losses-on-expectations-talks-to-ease-middle-east-supply-woes.html?...,Oil rises 2% as White House says no US-Iran talks happening,"Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomatic efforts by ot...","In this article Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomat...",Oil rises 2% as White House says no US-Iran talks happening Oil rises 2% as White House says no US-Iran talks happen...,Oil rises 2 percent as White House says no US Iran talks happening Oil rises 2 percent as White House says no US Ira...,oil rises 2 percent as white house says no us iran talks happening oil rises 2 percent as white house says no us ira...,oil rises percent white house us iran talks happening oil rises percent white house us iran talks happening oil pric...,oil rise percent white house u iran talk happen oil rise percent white house u iran talk happen oil price rise thurs...,2026-08-27 09:49:03,08/27/2026,09:49 AM,2026-08-27
2,https://www.cnbc.com/2026/07/21/jpmorgan-chase-ceo-jamie-dimon-market-risk.html?&qsearchterm=geopolitical risk,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,JPMorgan Chase CEO Jamie Dimon said investors are underestimating the risks facing the global economy and that he wo...,In this article JPMorgan ChaseCEOJamie Dimonsaid investors are underestimating the risks facing the global economy a...,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ... Jamie Dimon says...,Jamie Dimon says markets underestimate risks and he wouldn t buy stocks or Treasurys at current Jamie Dimon says mar...,jamie dimon says markets underestimate risks and he wouldn t buy stocks or treasurys at current jamie dimon says mar...,jamie dimon markets underestimate risks buy stocks treasurys current jamie dimon markets underestimate risks buy sto...,jamie dimon market underestimate risk buy stock treasurys current jamie dimon market underestimate risk buy stock tr...,2026-07-21 06:01:01,07/21/2026,06:01 AM,2026-07-21


# Visualisasi Word Cloud (Judul vs Teks Lengkap)
Visualisasi perbandingan distribusi kata kunci penting antara **Judul (Headline)** dan **Teks Lengkap (Full-Text)** setelah pembersihan tanda baca dan stopword removal.

In [28]:
!pip install wordcloud

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.2 -> 26.3
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [32]:
# Visualisasi Word Cloud untuk Judul dan Teks Lengkap setelah Stopword Removal
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from pathlib import Path
import html
import re

# Helper pembersih & filter kata setelah stopword removal
def get_clean_words(series):
    cleaned_words = []
    for text in series.dropna():
        t = html.unescape(str(text).lower())
        t = re.sub(r"http\S+|www\.\S+|<[^>]+>|[^a-zA-Z\s]", " ", t)
        words = [w for w in t.split() if w not in STOPWORDS_EN and len(w) > 2]
        cleaned_words.extend(words)
    return " ".join(cleaned_words)

print("Membuat teks gabungan untuk Word Cloud...")
title_words = get_clean_words(news["title"])
fulltext_words = get_clean_words(news["full_text"])

# 1. Word Cloud untuk Judul (Title)
wc_title = WordCloud(
    width=850,
    height=550,
    background_color="white",
    colormap="Blues",
    max_words=120,
    random_state=42
).generate(title_words)

# 2. Word Cloud untuk Teks Lengkap (Full-Text)
wc_fulltext = WordCloud(
    width=850,
    height=550,
    background_color="white",
    colormap="YlOrRd",
    max_words=120,
    random_state=42
).generate(fulltext_words)

# 3. Tampilkan Visualisasi Berdampingan
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].imshow(wc_title, interpolation="bilinear")
axes[0].set_title("Word Cloud - Judul Berita (Title)\n(Setelah Stopword Removal)", fontsize=15, fontweight="bold", pad=15)
axes[0].axis("off")

axes[1].imshow(wc_fulltext, interpolation="bilinear")
axes[1].set_title("Word Cloud - Teks Lengkap (Full-Text)\n(Setelah Stopword Removal)", fontsize=15, fontweight="bold", pad=15)
axes[1].axis("off")

plt.tight_layout(pad=2.0)

# Simpan gambar untuk kebutuhan laporan
figures_dir = Path("reports/figures")
figures_dir.mkdir(parents=True, exist_ok=True)
output_img_path = figures_dir / "wordcloud_comparison.png"
fig.savefig(output_img_path, dpi=300, bbox_inches="tight")
print(f"Visualisasi berhasil disimpan ke: {output_img_path}")

plt.show()


ModuleNotFoundError: No module named 'wordcloud'

# Visualisasi Word Cloud Setelah Process Full (Lemmatized)
Visualisasi Word Cloud untuk **Judul** dan **Teks Lengkap** setelah seluruh proses NLP (*stopword removal* dan *POS-aware lemmatization*) selesai. Kata-kata jamak dan berimbuhan telah dikembalikan ke bentuk kata dasarnya (misal: *prices* -> *price*, *markets* -> *market*).

In [ ]:
# Visualisasi Word Cloud Setelah Process Full (Cleaning + Stopwords + Lemmatization)
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from pathlib import Path
import html
import re

print("Membuat Word Cloud setelah process full (lemmatized)...")
title_token_docs = [
    [w for w in re.sub(r"http\S+|www\.\S+|<[^>]+>|[^a-zA-Z\s]", " ", html.unescape(str(t).lower())).split() if w not in STOPWORDS_EN and len(w) > 2]
    for t in news["title"].dropna()
]
title_lem_words = [w for doc in title_token_docs for w in lemmatize_tokens(doc)]

# Ambil teks lengkap yang sudah dilemmatisasi di notebook
if "text_lemmatized" in news.columns:
    fulltext_lem_text = " ".join(news["text_lemmatized"].dropna())
else:
    fulltext_lem_text = " ".join([lemmatizer.lemmatize(w) for w in fulltext_words.split()])

# 1. Word Cloud Judul (Full Process)
wc_title_full = WordCloud(
    width=850,
    height=550,
    background_color="white",
    colormap="Blues",
    max_words=120,
    random_state=42
).generate(" ".join(title_lem_words))

# 2. Word Cloud Teks Lengkap (Full Process)
wc_fulltext_full = WordCloud(
    width=850,
    height=550,
    background_color="white",
    colormap="YlOrRd",
    max_words=120,
    random_state=42
).generate(fulltext_lem_text)

# 3. Visualisasi Berdampingan
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].imshow(wc_title_full, interpolation="bilinear")
axes[0].set_title("Word Cloud: Judul Berita\n(Setelah Process Full / Lemmatized)", fontsize=15, fontweight="bold", pad=15)
axes[0].axis("off")

axes[1].imshow(wc_fulltext_full, interpolation="bilinear")
axes[1].set_title("Word Cloud: Teks Lengkap\n(Setelah Process Full / Lemmatized)", fontsize=15, fontweight="bold", pad=15)
axes[1].axis("off")

plt.tight_layout(pad=2.0)

# Simpan ke folder reports/figures
figures_dir = Path("reports/figures")
figures_dir.mkdir(parents=True, exist_ok=True)
save_path = figures_dir / "wordcloud_full_process.png"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
print(f"Gambar Word Cloud (Process Full) berhasil disimpan ke: {save_path}")

plt.show()


# Visualisasi Co-occurrence Network (Jaringan Keterhubungan Kata Kunci)
Visualisasi graf jaringan keterhubungan (**Co-occurrence Network**) antar kata kunci penting pada judul berita geopolitik dan ekonomi. Setiap *node* merepresentasikan kata kunci, *garis/edge* merepresentasikan frekuensi kemunculan bersama dalam satu judul, dan *warna* menunjukkan klaster topik komunitas (*topic cluster*) yang terdeteksi otomatis.

In [ ]:
# Visualisasi Co-occurrence Network untuk Judul Berita
import networkx as nx
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path
import html
import re

print("Menganalisis Co-occurrence Network...")

# 1. Ekstraksi kata per judul (setelah stopword removal & lemmatization)
docs = []
for t in news["title"].dropna():
    cleaned = re.sub(r"http\S+|www\.\S+|<[^>]+>|[^a-zA-Z\s]", " ", html.unescape(str(t).lower()))
    words = [w for w in cleaned.split() if w not in STOPWORDS_EN and len(w) > 2]
    lem_words = set(lemmatize_tokens(words))
    docs.append(lem_words)

# 2. Ambil top 35 kata teratas dan hitung pasangan ko-okurensi
word_counts = Counter(w for doc in docs for w in doc)
top_words = set([w for w, c in word_counts.most_common(35)])

co_occur = defaultdict(int)
for doc in docs:
    shared = [w for w in doc if w in top_words]
    for w1, w2 in combinations(sorted(shared), 2):
        co_occur[(w1, w2)] += 1

# 3. Bentuk Graph NetworkX
G = nx.Graph()
for word in top_words:
    G.add_node(word, count=word_counts[word])

# Hubungkan pasangan yang muncul bersama minimal 10 kali
min_co_occur = 10
for (w1, w2), weight in co_occur.items():
    if weight >= min_co_occur:
        G.add_edge(w1, w2, weight=weight)

# Bersihkan node tanpa relasi
G.remove_nodes_from(list(nx.isolates(G)))

# 4. Deteksi Klaster Komunitas & Pewarnaan
communities = list(nx.community.greedy_modularity_communities(G))
palette = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6", "#1abc9c"]
node_color_map = {}
for i, comm in enumerate(communities):
    color = palette[i % len(palette)]
    for node in comm:
        node_color_map[node] = color

node_colors = [node_color_map[n] for n in G.nodes()]
node_sizes = [word_counts[n] * 6.5 + 400 for n in G.nodes()]
edge_weights = [G[u][v]["weight"] for u, v in G.edges()]
edge_widths = [0.8 + (w / max(edge_weights)) * 5.0 for w in edge_weights]

# 5. Layout dan Visualisasi
pos = nx.spring_layout(G, k=0.55, iterations=80, seed=42)
plt.figure(figsize=(16, 11), dpi=200)

nx.draw_networkx_edges(G, pos, width=edge_widths, alpha=0.35, edge_color="#7f8c8d")
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors, alpha=0.88, edgecolors="#2c3e50", linewidths=1.8)
nx.draw_networkx_labels(
    G, pos,
    labels={n: n for n in G.nodes()},
    font_size=11, font_family="sans-serif", font_weight="bold", font_color="#1a1a1a",
    bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="none", alpha=0.75)
)

plt.title("Co-occurrence Network: Keterhubungan Kata Kunci Berita Geopolitik & Pasar\n(Warna = Klaster Topik, Ukuran Node = Frekuensi, Tebal Garis = Kekuatan Relasi)", fontsize=14, fontweight="bold", pad=25)
plt.axis("off")
plt.tight_layout()

# Simpan gambar otomatis untuk laporan
figures_dir = Path("reports/figures")
figures_dir.mkdir(parents=True, exist_ok=True)
output_path = figures_dir / "co_occurrence_network.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
print(f"Visualisasi Co-occurrence Network berhasil disimpan ke: {output_path}")

plt.show()
